In [1]:
# Cargar Bronze desde Supabase y preparar Silver/Gold
import sys
import os

import pandas as pd

sys.path.insert(0, os.path.abspath("../.secrets"))
from db_config import get_connection

conn = get_connection()
print("Conectado a Supabase")

tablas = [
    "dim_asesor",
    "fact_reclutamiento",
    "fact_rendimiento_mensual",
    "fact_capacitacion",
    "fact_calidad",
    "fact_incidencias",
    "fact_adherencia",
    "fact_clima",
]

data_bronze = {}
for t in tablas:
    data_bronze[t] = pd.read_sql(f"SELECT * FROM bronze.{t}", conn)

print("Tablas bronze cargadas:", list(data_bronze.keys()))

Conectado a Supabase


C:\Users\Asus\AppData\Local\Temp\ipykernel_16660\1750126979.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_bronze[t] = pd.read_sql(f"SELECT * FROM bronze.{t}", conn)


Tablas bronze cargadas: ['dim_asesor', 'fact_reclutamiento', 'fact_rendimiento_mensual', 'fact_capacitacion', 'fact_calidad', 'fact_incidencias', 'fact_adherencia', 'fact_clima']


## Capa Silver
Limpieza: tipos, duplicados, nulos y normalización de textos.

In [2]:
def limpiar_silver(df):
    """Limpieza genérica para capa silver."""
    out = df.copy()
    # Eliminar duplicados
    out = out.drop_duplicates()
    # Cadenas: strip y normalizar espacios
    for c in out.select_dtypes(include=["object"]).columns:
        out[c] = out[c].astype(str).str.strip().replace("nan", "")
    # Numéricos: rellenar NaN con 0 donde sea coherente
    for c in out.select_dtypes(include=["number"]).columns:
        if out[c].dtype in ["float64", "int64"]:
            out[c] = out[c].fillna(0)
    return out

data_silver = {nombre: limpiar_silver(df) for nombre, df in data_bronze.items()}
print("Silver generado:", list(data_silver.keys()))

Silver generado: ['dim_asesor', 'fact_reclutamiento', 'fact_rendimiento_mensual', 'fact_capacitacion', 'fact_calidad', 'fact_incidencias', 'fact_adherencia', 'fact_clima']


C:\Users\Asus\AppData\Local\Temp\ipykernel_16660\3655260375.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in out.select_dtypes(include=["object"]).columns:
C:\Users\Asus\AppData\Local\Temp\ipykernel_16660\3655260375.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

## Capa Gold
Agregados de negocio: resumen por asesor y métricas consolidadas.

In [3]:
# Gold: dimensión asesor (limpia, lista para reporting)
gold_dim_asesor = data_silver["dim_asesor"].copy()

# Gold: resumen por asesor a partir de cada fact (evitando columnas duplicadas)
def agregar_por_asesor(nombre_fact, sufijo=None):
    df = data_silver.get(nombre_fact)
    if df is None or df.empty or "id_asesor" not in df.columns:
        return None
    cols_num = [c for c in df.select_dtypes(include=["number"]).columns if c != "id_asesor"]
    if not cols_num:
        out = df.groupby("id_asesor", as_index=False).size().rename(columns={"size": "registros"})
    else:
        out = df.groupby("id_asesor", as_index=False)[cols_num].sum()
    if sufijo:
        out = out.rename(columns={c: f"{c}_{sufijo}" for c in out.columns if c != "id_asesor"})
    return out

gold_resumen_asesor = data_silver["dim_asesor"][["id_asesor"]].drop_duplicates()
for fact in ["fact_rendimiento_mensual", "fact_capacitacion", "fact_calidad", "fact_incidencias", "fact_adherencia", "fact_clima", "fact_reclutamiento"]:
    agg = agregar_por_asesor(fact, sufijo=fact.replace("fact_", ""))
    if agg is not None and not agg.empty:
        gold_resumen_asesor = gold_resumen_asesor.merge(agg, on="id_asesor", how="left")

data_gold = {
    "dim_asesor": gold_dim_asesor,
    "resumen_por_asesor": gold_resumen_asesor,
}
print("Gold generado:", list(data_gold.keys()))

Gold generado: ['dim_asesor', 'resumen_por_asesor']


## Crear schemas y subir Silver y Gold a Supabase

In [4]:
from sqlalchemy import create_engine

# URI desde la misma config (ajusta si usas variables de entorno)
import os
from dotenv import load_dotenv
load_dotenv(os.path.abspath("../.env"))
if not os.path.exists(os.path.abspath("../.env")):
    load_dotenv()

user = os.getenv("DB_USER", "postgres.zuxjvpqgdpprzostfiuf")
password = os.getenv("DB_PASSWORD", "xmatansa123")
host = os.getenv("DB_HOST", "aws-0-us-west-2.pooler.supabase.com")
port = os.getenv("DB_PORT", "5432")
db = os.getenv("DB_NAME", "postgres")
uri = f"postgresql://{user}:{password}@{host}:{port}/{db}"
engine = create_engine(uri)

cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
cur.execute("CREATE SCHEMA IF NOT EXISTS gold;")
conn.commit()
cur.close()
print("Schemas silver y gold creados (o ya existían).")

Schemas silver y gold creados (o ya existían).


In [5]:
# Subir tablas Silver
for nombre, df in data_silver.items():
    df.to_sql(nombre, engine, schema="silver", if_exists="replace", index=False)
    print(f"Silver: {nombre} -> silver.{nombre}")

# Subir tablas Gold
for nombre, df in data_gold.items():
    df.to_sql(nombre, engine, schema="gold", if_exists="replace", index=False)
    print(f"Gold: {nombre} -> gold.{nombre}")

print("\nSilver y Gold subidos a Supabase correctamente.")

Silver: dim_asesor -> silver.dim_asesor
Silver: fact_reclutamiento -> silver.fact_reclutamiento
Silver: fact_rendimiento_mensual -> silver.fact_rendimiento_mensual
Silver: fact_capacitacion -> silver.fact_capacitacion
Silver: fact_calidad -> silver.fact_calidad
Silver: fact_incidencias -> silver.fact_incidencias
Silver: fact_adherencia -> silver.fact_adherencia
Silver: fact_clima -> silver.fact_clima
Gold: dim_asesor -> gold.dim_asesor
Gold: resumen_por_asesor -> gold.resumen_por_asesor

Silver y Gold subidos a Supabase correctamente.
